# Twitter Sentiment Analysis — 2020 US Presidential Election

Analysis of ~1.72M tweets collected between October 15 and November 8, 2020, covering the weeks before, during, and after the election (November 3).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import string
import re
import warnings

import nltk
import spacy
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import TweetTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import sys
sys.path.append("../src")
from preprocessing import load_and_filter_tweets, prepare_for_sentiment

warnings.filterwarnings("ignore")
nltk.download("vader_lexicon", quiet=True)
pio.renderers.default = "notebook"

plt.rcParams["figure.dpi"] = 120
ELECTION_DAY = pd.Timestamp("2020-11-03")

## 1. Data Loading

Load the raw tweets and the pre-computed ABSA sentiment output (produced by `src/absa_sentiment.py`).

In [ ]:
# Raw English tweets
tweets_df = load_and_filter_tweets("../data/tweets.csv")
print(f"English tweets: {len(tweets_df):,}")
tweets_df.head(2)

In [ ]:
# ABSA sentiment output
sents_df = pd.read_csv("../data/sentiment.csv", engine="python", dtype={"tweet_id": str})
sents_df.columns = [
    "tweet_id", "tweet_cleaned", "tweet_original", "targets",
    "Biden.sentiment", "Trump.sentiment", "Biden.confidence", "Trump.confidence",
]
sents_df["tweet_id"] = pd.to_numeric(sents_df["tweet_id"], errors="coerce")
sents_df.dropna(subset=["targets"], inplace=True)
print(f"Tweets with sentiment labels: {len(sents_df):,}")
sents_df.head(2)

In [ ]:
# Merge on tweet_id, add candidate flags and temporal features
df = pd.merge(sents_df, tweets_df, on="tweet_id")
df["hasTrump"] = df["targets"].apply(lambda x: "Trump" in str(x))
df["hasBiden"] = df["targets"].apply(lambda x: "Biden" in str(x))
df["only_trump"] = df["hasTrump"] & ~df["hasBiden"]
df["only_biden"] = df["hasBiden"] & ~df["hasTrump"]
df["has_both"]   = df["hasTrump"] & df["hasBiden"]
df["liked"]      = df["likes"] > 0
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df["created_date"] = df["created_at"].dt.date

print(f"Merged dataset: {len(df):,} rows")
print(df[["only_trump","only_biden","has_both"]].mean().apply(lambda x: f"{x:.1%}").rename("share"))
df.head(2)

## 2. VADER Sentiment Analysis

VADER is a lexicon-based tool designed for social media text. It outputs a compound score in **[-1, 1]**.

In [ ]:
sia = SentimentIntensityAnalyzer()
df["vader_score"] = df["tweet_cleaned"].apply(lambda x: sia.polarity_scores(str(x))["compound"])
df["vader_score"].describe()

### 2.1 Sentiment distribution by candidate

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for_who = df["only_trump"].map({True: "Trump"}).fillna(
          df["only_biden"].map({True: "Biden"}).fillna("Both"))
df["for_who"] = for_who

sns.boxplot(x="for_who", y="vader_score", data=df, order=["Biden", "Both", "Trump"],
            palette={"Biden": "steelblue", "Both": "mediumpurple", "Trump": "tomato"}, ax=ax)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Tweet Sentiment by Candidate (VADER)")
ax.set_xlabel("For Who")
ax.set_ylabel("VADER Compound Score")
plt.tight_layout()
plt.show()

### 2.2 Sentiment over time

In [ ]:
trump_daily = df[df["only_trump"]].groupby("created_date")["vader_score"].mean()
biden_daily = df[df["only_biden"]].groupby("created_date")["vader_score"].mean()
both_daily  = df[df["has_both"]].groupby("created_date")["vader_score"].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(trump_daily.index, trump_daily.values, label="Trump", color="tomato")
ax.plot(biden_daily.index, biden_daily.values, label="Biden", color="steelblue")
ax.plot(both_daily.index,  both_daily.values,  label="Both",  color="mediumpurple")
ax.axvline(ELECTION_DAY.date(), color="black", linestyle="--", linewidth=1.5, label="Election Day")
ax.set_title("Average Tweet Sentiment Over Time by Candidate (VADER)")
ax.set_xlabel("Date")
ax.set_ylabel("Average VADER Compound Score")
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 2.3 State-level sentiment vs. vote share

In [ ]:
# Election results dataset (Kaggle)
results = pd.read_csv("../data/voting.csv")

# Average sentiment per state (only-candidate tweets)
trump_state = df[df["only_trump"]].groupby("state")["vader_score"].mean().rename("trump_sentiment")
biden_state = df[df["only_biden"]].groupby("state")["vader_score"].mean().rename("biden_sentiment")

state_df = (results
    .merge(trump_state.reset_index(), on="state", how="left")
    .merge(biden_state.reset_index(), on="state", how="left"))

# Pearson correlations
trump_corr = state_df["trump_sentiment"].corr(state_df["trump_pct"])
biden_corr = state_df["biden_sentiment"].corr(state_df["biden_pct"])
print(f"Trump — sentiment vs vote %: r = {trump_corr:.2f}")
print(f"Biden — sentiment vs vote %: r = {biden_corr:.2f}")

In [ ]:
plot_df = state_df[state_df["state_abr"] != "DC"]  # exclude DC outlier

for candidate, col, scale in [("Biden", "biden", "Blues"), ("Trump", "trump", "Reds")]:
    sentiment_col = f"{col}_sentiment"
    pct_col       = f"{col}_pct"
    vmin = plot_df[pct_col].min()
    vmax = plot_df[pct_col].max()

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "choropleth"}, {"type": "choropleth"}]],
        subplot_titles=(f"{candidate} Tweet Sentiment", f"{candidate} Vote %"),
    )
    for col_idx, (z_col, title, colorbar_x, zmin, zmax) in enumerate([
        (sentiment_col, "Sentiment", 0.45, None, None),
        (pct_col,       "Vote %",    1.00, vmin,  vmax),
    ], start=1):
        fig.add_trace(go.Choropleth(
            locations=plot_df["state_abr"], z=plot_df[z_col],
            locationmode="USA-states", colorscale=scale,
            colorbar=dict(title=title, x=colorbar_x),
            **({"zmin": zmin, "zmax": zmax} if zmin is not None else {}),
        ), row=1, col=col_idx)
        fig.update_geos(scope="usa", row=1, col=col_idx)

    fig.update_layout(
        title_text=f"{candidate}: Twitter Sentiment vs Vote % by State (VADER)",
        height=500, width=1000,
    )
    fig.show()

## 3. ABSA Sentiment Analysis

Aspect-Based Sentiment Analysis (PyABSA) assigns sentiment per candidate within each tweet, handling tweets that mention both.

### 3.1 Sentiment distribution

In [ ]:
for candidate in ["Biden", "Trump"]:
    col = f"{candidate}.sentiment"
    counts = df[df["has" + candidate]][col].value_counts(normalize=True)
    print(f"\n{candidate}:")
    print(counts.apply(lambda x: f"{x:.1%}"))

In [ ]:
# Quantify ABSA sentiment: positive → +confidence, neutral → 0, negative → -confidence
def absa_score(row, candidate):
    sent = row[f"{candidate}.sentiment"]
    conf = row[f"{candidate}.confidence"]
    if sent == "Positive":
        return conf
    elif sent == "Negative":
        return -conf
    elif sent == "Neutral":
        return 0.0
    return np.nan

df["absa_score_biden"] = df.apply(lambda r: absa_score(r, "Biden"), axis=1)
df["absa_score_trump"] = df.apply(lambda r: absa_score(r, "Trump"), axis=1)

print(f"Mean ABSA score — Biden: {df['absa_score_biden'].mean():.4f}")
print(f"Mean ABSA score — Trump: {df['absa_score_trump'].mean():.4f}")

### 3.2 Sentiment over time (ABSA)

In [ ]:
trump_absa_daily = df.groupby("created_date")["absa_score_trump"].mean()
biden_absa_daily = df.groupby("created_date")["absa_score_biden"].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(trump_absa_daily.index, trump_absa_daily.values, label="Trump", color="darkred")
ax.plot(biden_absa_daily.index, biden_absa_daily.values, label="Biden", color="darkblue")
ax.axvline(ELECTION_DAY.date(), color="black", linestyle="--", linewidth=1.5, label="Election Day")
ax.set_title("Average Tweet Sentiment Over Time by Candidate (ABSA)")
ax.set_xlabel("Date")
ax.set_ylabel("Average ABSA Score")
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.3 VADER vs. ABSA correlation

In [ ]:
biden_only = df[df["only_biden"]][["absa_score_biden", "vader_score"]].dropna()
trump_only = df[df["only_trump"]][["absa_score_trump", "vader_score"]].dropna()

print("VADER vs ABSA Pearson correlation:")
print(f"  Biden-only tweets: r = {biden_only.corr().iloc[0,1]:.2f}")
print(f"  Trump-only tweets: r = {trump_only.corr().iloc[0,1]:.2f}")

### 3.4 ABSA state-level sentiment vs. vote share

In [ ]:
trump_state_absa = df[df["only_trump"]].groupby("state")["absa_score_trump"].mean().rename("trump_sentiment")
biden_state_absa = df[df["only_biden"]].groupby("state")["absa_score_biden"].mean().rename("biden_sentiment")

state_absa_df = (results
    .merge(trump_state_absa.reset_index(), on="state", how="left")
    .merge(biden_state_absa.reset_index(), on="state", how="left"))

trump_corr_absa = state_absa_df["trump_sentiment"].corr(state_absa_df["trump_pct"])
biden_corr_absa = state_absa_df["biden_sentiment"].corr(state_absa_df["biden_pct"])
print(f"ABSA — Trump: r = {trump_corr_absa:.2f}")
print(f"ABSA — Biden: r = {biden_corr_absa:.2f}")

In [ ]:
# ABSA choropleth maps (same helper as VADER section above)
plot_absa = state_absa_df[state_absa_df["state_abr"] != "DC"]

for candidate, col, scale in [("Biden", "biden", "Blues"), ("Trump", "trump", "Reds")]:
    sentiment_col = f"{col}_sentiment"
    pct_col       = f"{col}_pct"
    vmin = plot_absa[pct_col].min()
    vmax = plot_absa[pct_col].max()

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "choropleth"}, {"type": "choropleth"}]],
        subplot_titles=(f"{candidate} Tweet Sentiment (ABSA)", f"{candidate} Vote %"),
    )
    for col_idx, (z_col, title, colorbar_x, zmin, zmax) in enumerate([
        (sentiment_col, "Sentiment", 0.45, None, None),
        (pct_col,       "Vote %",    1.00, vmin,  vmax),
    ], start=1):
        fig.add_trace(go.Choropleth(
            locations=plot_absa["state_abr"], z=plot_absa[z_col],
            locationmode="USA-states", colorscale=scale,
            colorbar=dict(title=title, x=colorbar_x),
            **({"zmin": zmin, "zmax": zmax} if zmin is not None else {}),
        ), row=1, col=col_idx)
        fig.update_geos(scope="usa", row=1, col=col_idx)

    fig.update_layout(
        title_text=f"{candidate}: Twitter Sentiment vs Vote % by State (ABSA)",
        height=500, width=1000,
    )
    fig.show()

## 4. Tweet Popularity Prediction

Binary classification: did a tweet receive **at least one like**? We use TF-IDF features + Multinomial Naïve Bayes.

In [ ]:
IMPORTANT_USERS = {"@realDonaldTrump", "@JoeBiden"}
tknzr = TweetTokenizer(preserve_case=True, reduce_len=True, strip_handles=False)

def tweet_tokenizer(text: str) -> list[str]:
    # Replace links
    text = re.sub(r"http\S+|www\S+|https\S+", "https", text)
    # Remove irrelevant mentions
    text = re.sub(r"@\w+", lambda m: m.group() if m.group() in IMPORTANT_USERS else "", text)
    tokens = tknzr.tokenize(text)
    # Remove punctuation and noise tokens
    noise = set(string.punctuation) | {"\uFE0F", "..", "m", "u"}
    return [t for t in tokens if t not in noise]

X = df["tweet_original"].fillna("")
y = df["liked"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(
    tokenizer=tweet_tokenizer,
    stop_words="english",
    ngram_range=(1, 1),
    max_features=20_000,
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

clf = MultinomialNB(alpha=1.0)
clf.fit(X_train_vec, y_train)
y_pred = clf.predict(X_test_vec)

print(f"Training accuracy: {accuracy_score(y_train, clf.predict(X_train_vec)):.3f}")
print(f"Testing  accuracy: {accuracy_score(y_test, y_pred):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["Not Liked", "Liked"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Liked", "Liked"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Naïve Bayes Tweet Classifier")
plt.tight_layout()
plt.show()

### 4.1 Most predictive words

In [ ]:
feature_names = vectorizer.get_feature_names_out()
log_probs     = clf.feature_log_prob_
log_odds      = log_probs[1] - log_probs[0]  # liked vs not-liked

# Restrict to the top-100 most frequent tokens for interpretability
token_counts = np.asarray(X_train_vec.sum(axis=0)).flatten()
top100_idx   = set(np.argsort(token_counts)[::-1][:100])
sort_odds     = np.argsort(log_odds)
filtered      = [i for i in sort_odds if i in top100_idx]

top_liked_idx    = filtered[-15:][::-1]
top_disliked_idx = filtered[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(feature_names[top_liked_idx][::-1], log_odds[top_liked_idx][::-1], color="steelblue")
axes[0].set_title("15 Words Most Indicative of Liked Tweets")
axes[0].set_xlabel("Log-Odds (Liked vs Not Liked)")

axes[1].barh(feature_names[top_disliked_idx][::-1], log_odds[top_disliked_idx][::-1], color="tomato")
axes[1].set_title("15 Words Most Indicative of Not-Liked Tweets")
axes[1].set_xlabel("Log-Odds (Liked vs Not Liked)")

plt.tight_layout()
plt.show()

## 5. Summary

| | Biden r | Trump r |
|---|---|---|
| VADER (state-level) | 0.38 | 0.34 |
| ABSA (state-level)  | 0.21 | 0.13 |

**Key findings:**
- Both VADER and ABSA agree: sentiment towards Biden is consistently more positive than towards Trump.
- Sentiment remained relatively stable in the weeks before the election, became more positive in the days immediately before it, peaked just after the result, then dropped sharply.
- VADER correlates more strongly with actual vote shares than ABSA, likely because ABSA was used zero-shot on a domain it was not fine-tuned for, causing it to over-predict neutral sentiment.
- The popularity model (61% accuracy) shows that real-time, action-oriented language (*electionnight*, *votes*, *won*) drives engagement, while generic political vocabulary (*democrats*, *america*, *election*) does not.